# Sanity Checks on Simulation Runs

Before running any analysis, this notebook verifies that the raw simulation
output actually implements the experimental design described in the paper's
**Setup** section: that networks look as intended, that role/model
assignments are randomized where they should be, and that randomization was
applied *consistently* where it should be held fixed (e.g., the same seed
should give the same network+roles across statements).

None of these checks produce a paper figure or number; they are a
data-integrity gate that should pass before `X1_data.ipynb` is trusted.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl
from cmcrameri import cm

sys.path.insert(0, str(Path.cwd().parent))

from src.analysis.loaders import load_adjacency_matrix, load_agents_data, load_runs_metadata

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 10)
plt.rcParams["font.size"] = 11

df_total = pl.from_pandas(load_runs_metadata(Path.cwd().parent / "data" / "outputs" / "runs"))
df_runs = df_total.filter(pl.col("validated") & pl.col("completed")).to_pandas()

print(f"Total validated runs: {len(df_runs)}")
print("\nRuns by setting:")
print(df_runs.groupby("setting").size())
print("\nRuns by graph type:")
print(df_runs.groupby("graph_type").size())


In [ ]:
# Each (setting, statement) combination should have the same number of runs
# (4 scenarios x 2 network types x 8 realizations x 20 statements = 1,280 runs total).
df_runs.groupby(["setting", "statement_id"]).agg("size").unstack(fill_value=0)


## Flagging incomplete / unvalidated runs

`move_invalid_runs` relocates any run directory that failed completion or
post-hoc validation into a separate `incomplete_runs/` folder, so downstream
notebooks only ever see the clean set. Default to `dry_run=True` here: it
reports what *would* move without touching the filesystem. Set `dry_run=False`
only when you are ready to actually clean up a local `data/outputs/runs/`.


In [ ]:
from src.utils import move_invalid_runs

stats = move_invalid_runs(
    runs_dir=Path.cwd().parent / "data" / "outputs" / "runs",
    incomplete_dir=Path.cwd().parent / "data" / "outputs" / "incomplete_runs",
    dry_run=True,
)

print("Summary (dry run - nothing was moved):")
print(f"  Incomplete runs:  {stats['incomplete']}")
print(f"  Unvalidated runs: {stats['unvalidated']}")
print(f"  Total to move:    {stats['total']}")


## 1. Network topology

Visually confirm that ER and WS network realizations look structurally distinct.

In [ ]:
import networkx as nx
import numpy as np

unique_graphs = df_runs.drop_duplicates(["seed", "graph_type"]).sort_values("graph_type")

adjacency, graph_id, degrees = [], [], []
for _, run in unique_graphs.iterrows():
    A, _ = load_adjacency_matrix(run["run_path"])
    adjacency.append(A)
    graph_id.append(f"{run['graph_type']}-{run['seed']}")
    degrees.extend(np.sum(A, axis=1))

er_graphs = [(A, gid) for A, gid in zip(adjacency, graph_id) if "er" in gid.lower()]
ws_graphs = [(A, gid) for A, gid in zip(adjacency, graph_id) if "ws" in gid.lower() or "watts" in gid.lower()]


def draw_graph(ax, A, gid, min_degree=0, max_degree=10):
    G = nx.from_numpy_array(A)
    pos = nx.kamada_kawai_layout(G)
    node_degrees = np.array([d for _, d in G.degree()])

    nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.35, width=0.8, edge_color="#888888")
    nodes = nx.draw_networkx_nodes(
        G, pos, ax=ax, node_size=60, node_color=node_degrees,
        cmap=cm.acton_r, vmin=min_degree, vmax=max_degree,
    )

    seed = gid.split("-")[-1]
    ax.set_title(
        f"seed={seed} | $n$={G.number_of_nodes()}, $m$={G.number_of_edges()}\n"
        f"$\\bar{{k}}$={node_degrees.mean():.1f}, $\\sigma_k$={node_degrees.std():.1f}, "
        f"$C$={nx.average_clustering(G):.2f}",
        fontsize=7.5, pad=2,
    )
    ax.axis("off")
    return nodes


fig, axes = plt.subplots(8, 2, figsize=(15, 28))
fig.suptitle("Network Topologies: ER (left) vs WS (right)", fontsize=13, fontweight="bold")
for ax, title in zip(axes[0], ["Erdős–Rényi", "Watts–Strogatz"]):
    ax.set_title(title, fontsize=11, fontweight="bold", pad=2)

mappables = []
max_degree = max(degrees)
for row, ((A_er, gid_er), (A_ws, gid_ws)) in enumerate(zip(er_graphs, ws_graphs)):
    mappables.append(draw_graph(axes[row, 0], A_er, gid_er, 0, max_degree))
    mappables.append(draw_graph(axes[row, 1], A_ws, gid_ws, 0, max_degree))

cbar_ax = fig.add_axes([0.2, -0.01, 0.6, 0.008])
fig.colorbar(mappables[0], cax=cbar_ax, orientation="horizontal", label="Node Degree")
plt.tight_layout()
plt.show()


## 2. Adjacency matrices

Same realizations as above, shown as heatmaps.

In [ ]:
fig, axes = plt.subplots(8, 2, figsize=(10, 30), constrained_layout=True)
fig.suptitle("Adjacency Matrices: ER (left) vs WS (right)", fontsize=13, fontweight="bold")
for ax, title in zip(axes[0], ["Erdős–Rényi", "Watts–Strogatz"]):
    ax.set_title(title, fontsize=11, fontweight="bold", pad=8)


def draw_matrix(ax, A, gid):
    G = nx.from_numpy_array(A)
    node_degrees = np.array([d for _, d in G.degree()])
    seed = gid.split("-")[-1]

    sns.heatmap(
        A, ax=ax, cbar=False, cmap=cm.acton_r, vmin=0, vmax=1, square=True,
        linewidths=0.3, linecolor="#dddddd", xticklabels=False, yticklabels=False,
    )
    ax.set_title(
        f"seed={seed} | $n$={G.number_of_nodes()}, $m$={G.number_of_edges()}\n"
        f"$\\bar{{k}}$={node_degrees.mean():.1f}, $\\sigma_k$={node_degrees.std():.1f}, "
        f"$C$={nx.average_clustering(G):.2f}",
        fontsize=7.5, pad=3,
    )


for row, ((A_er, gid_er), (A_ws, gid_ws)) in enumerate(zip(er_graphs, ws_graphs)):
    draw_matrix(axes[row, 0], A_er, gid_er)
    draw_matrix(axes[row, 1], A_ws, gid_ws)
plt.show()


## 3. Role and model connectivity

Confirm that, within a single run, roles and underlying models are actually
wired onto the network the way each scenario intends (e.g., in `experts`,
role labels should match each agent's underlying model per `Appendix D`'s
role-model pairing).


In [ ]:
MODEL_TO_IDS = {
    "llama-assistant": 0, "llama-base": 1, "llama-biomed": 2, "llama-chemist": 3,
    "llama-coder": 4, "llama-cyber": 5, "llama-doc": 6, "llama-finance": 7,
    "llama-hermes": 8, "llama-lexicographer": 9, "llama-linguist": 10,
    "llama-openmath": 11, "llama-roleplay": 12, "llama-scholar": 13, "llama-user": 14,
}

ROLE_TO_IDS = {
    "Academic Scholar": 0, "Assistant": 1, "Biomedical Researcher": 2, "Chemist": 3,
    "Clinical Physician": 4, "Cybersecurity Analyst": 5, "Financial Analyst": 6,
    "Human Participant": 7, "LLM": 8, "Language Mediator": 9, "Lexicographer": 10,
    "Mathematician": 11, "Online Friend": 12, "Software Engineer": 13,
    "Storyteller": 14, "Strategic Planner": 15,
}


def build_connection_matrix(A, agent_labels, labels_to_ids):
    """Count edges between each pair of (role or model) categories."""
    n_labels = len(labels_to_ids)
    conn_matrix = np.zeros((n_labels, n_labels), dtype=int)

    n_agents = len(agent_labels)
    for i in range(n_agents):
        for j in range(i + 1, n_agents):
            if A[i, j] > 0:
                id_i = labels_to_ids.get(agent_labels[i], -1)
                id_j = labels_to_ids.get(agent_labels[j], -1)
                if id_i != -1 and id_j != -1:
                    conn_matrix[id_i, id_j] += 1
                    conn_matrix[id_j, id_i] += 1
    return conn_matrix


def plot_connectivity_for_setting(setting_to_show: str, exclude_settings: list[str]):
    unique_df = df_runs.drop_duplicates(["seed", "graph_type", "setting"]).sort_values("graph_type")
    unique_df = unique_df[~unique_df["setting"].isin(exclude_settings)]

    role_labels = [k for k, _ in sorted(ROLE_TO_IDS.items(), key=lambda x: x[1])]
    model_labels = [k for k, _ in sorted(MODEL_TO_IDS.items(), key=lambda x: x[1])]

    n_runs = len(unique_df)
    fig, axes = plt.subplots(n_runs, 2, figsize=(16, 5 * n_runs))
    if n_runs == 1:
        axes = axes.reshape(1, -1)

    for row_idx, (_, row) in enumerate(unique_df.iterrows()):
        A, _ = load_adjacency_matrix(row["run_path"])
        agent_data = load_agents_data(row["run_path"])
        agent_ids = sorted(agent_data["models"].keys())
        models_list = [agent_data["models"][aid] for aid in agent_ids]
        roles_list = [agent_data["roles"][aid] for aid in agent_ids]

        role_conn = build_connection_matrix(A, roles_list, ROLE_TO_IDS)
        model_conn = build_connection_matrix(A, models_list, MODEL_TO_IDS)

        sns.heatmap(role_conn, annot=True, fmt="d", cmap="YlGnBu", xticklabels=role_labels,
                    yticklabels=role_labels, ax=axes[row_idx, 0], cbar_kws={"label": "Count"})
        axes[row_idx, 0].set_title(f"Role {row['setting']}: {row['graph_type']} | seed {row['seed']}")
        axes[row_idx, 0].set_xticklabels(role_labels, rotation=45, ha="right")

        sns.heatmap(model_conn, annot=True, fmt="d", cmap="YlOrRd", xticklabels=model_labels,
                    yticklabels=model_labels, ax=axes[row_idx, 1], cbar_kws={"label": "Count"})
        axes[row_idx, 1].set_title(f"Model {row['setting']}: {row['graph_type']} | seed {row['seed']}")
        axes[row_idx, 1].set_xticklabels(model_labels, rotation=45, ha="right")

    plt.tight_layout()
    plt.show()


#### `experts` setting (Scenario IV: matched roles)

In [ ]:
plot_connectivity_for_setting(
    "experts", exclude_settings=["base_llms", "random_roles", "random_experts"]
)


#### `random_experts` setting (Scenario III: random roles)

In [ ]:
plot_connectivity_for_setting(
    "random_experts", exclude_settings=["base_llms", "random_roles", "experts"]
)


## 4. Randomization checker

Verify the four properties the experimental design (`§Scenarios and Simulations`)
relies on:

- **(A)** Distinct seeds actually produce distinct network realizations.
- **(B)** In `random_roles`, role labels are shuffled across seeds (not fixed).
- **(C)** In `random_experts`, both roles *and* underlying models are shuffled
  across seeds.
- **(D)** For a fixed seed, the network/roles/models are held constant across
  the 20 discussion statements (statement is the only thing that varies).


In [ ]:
import hashlib

import pandas as pd

TARGET_SETTINGS = ["random_roles", "random_experts", "experts"]


def hash_array(arr: np.ndarray) -> str:
    return hashlib.sha256(np.ascontiguousarray(arr).tobytes()).hexdigest()


def hash_tuple(values: tuple[str, ...]) -> str:
    return hashlib.sha256(str(values).encode("utf-8")).hexdigest()


def load_run_signatures(run_path: str) -> dict[str, str]:
    """Stable hash signatures for a run's graph, role assignment, and model assignment."""
    A, _ = load_adjacency_matrix(run_path)
    agent_data = load_agents_data(run_path)
    agent_ids = sorted(agent_data["models"].keys())
    return {
        "graph_sig": hash_array(A),
        "roles_sig": hash_tuple(tuple(agent_data["roles"][aid] for aid in agent_ids)),
        "models_sig": hash_tuple(tuple(agent_data["models"][aid] for aid in agent_ids)),
    }


rows = []
for _, row in df_runs[df_runs["setting"].isin(TARGET_SETTINGS)].iterrows():
    sigs = load_run_signatures(row["run_path"])
    rows.append({
        "setting": row["setting"], "graph_type": row["graph_type"],
        "statement_id": row["statement_id"], "seed": int(row["seed"]),
        "run_path": row["run_path"], **sigs,
    })
sig_df = pd.DataFrame(rows)
print(f"Loaded signatures for {len(sig_df)} runs in {TARGET_SETTINGS}")

# (A) Graph realizations must differ across seeds within a (setting, graph_type, statement) group.
graph_diversity = (
    sig_df.groupby(["setting", "graph_type", "statement_id"])
    .agg(n_runs=("graph_sig", "size"), n_unique_graphs=("graph_sig", "nunique"))
    .reset_index()
)
a_fail = graph_diversity[
    (graph_diversity["n_runs"] > 1)
    & (graph_diversity["n_unique_graphs"] != graph_diversity["n_runs"])
]

# (B) random_roles: role assignment must vary across seeds.
rr = sig_df[sig_df["setting"] == "random_roles"]
rr_roles = (
    rr.groupby(["graph_type", "statement_id"])
    .agg(n_runs=("roles_sig", "size"), n_unique_role_maps=("roles_sig", "nunique"))
    .reset_index()
)
b_fail = (
    rr_roles[(rr_roles["n_runs"] > 1) & (rr_roles["n_unique_role_maps"] <= 1)]
    if len(rr_roles) else pd.DataFrame()
)

# (C) random_experts: both role and model assignment must vary across seeds.
re_ = sig_df[sig_df["setting"] == "random_experts"]
re_mix = (
    re_.groupby(["graph_type", "statement_id"])
    .agg(n_runs=("run_path", "size"), n_unique_role_maps=("roles_sig", "nunique"),
         n_unique_model_maps=("models_sig", "nunique"))
    .reset_index()
)
c_fail = (
    re_mix[(re_mix["n_runs"] > 1)
           & ((re_mix["n_unique_role_maps"] <= 1) | (re_mix["n_unique_model_maps"] <= 1))]
    if len(re_mix) else pd.DataFrame()
)

# (D) For a fixed (setting, graph_type, seed), graph/roles/models must be IDENTICAL across statements.
seed_consistency_rows = []
for (setting, graph_type, seed), group in sig_df.groupby(["setting", "graph_type", "seed"]):
    if group["statement_id"].nunique() <= 1:
        continue
    seed_consistency_rows.append({
        "setting": setting, "graph_type": graph_type, "seed": int(seed),
        "n_statements": group["statement_id"].nunique(),
        "graph_fixed": group["graph_sig"].nunique() == 1,
        "roles_fixed": group["roles_sig"].nunique() == 1,
        "models_fixed": group["models_sig"].nunique() == 1,
    })
seed_consistency = pd.DataFrame(seed_consistency_rows)
d_fail = (
    seed_consistency[~seed_consistency["graph_fixed"] | ~seed_consistency["roles_fixed"]
                      | ~seed_consistency["models_fixed"]]
    if len(seed_consistency) else pd.DataFrame()
)

print("\n=== Randomization check summary ===")
print(f"(A) graph-uniqueness-across-seeds failures:        {len(a_fail)}")
print(f"(B) random_roles role-randomization failures:      {len(b_fail)}")
print(f"(C) random_experts role/model-randomization fails: {len(c_fail)}")
print(f"(D) same-seed cross-statement consistency fails:   {len(d_fail)}")

for name, fail_df in [("A", a_fail), ("B", b_fail), ("C", c_fail), ("D", d_fail)]:
    if len(fail_df) > 0:
        print(f"\n{name} failure examples:")
        print(fail_df.head(10).to_string(index=False))

if all(len(f) == 0 for f in [a_fail, b_fail, c_fail, d_fail]):
    print("\nAll checks passed for available runs.")


## 5. `random_experts` connectivity uniqueness

A stricter version of check (C): confirm that the role-role and model-model
*connectivity matrices themselves* (not just the assignment) are distinct
across seeds, and that both roles and models actually connect across
categories (i.e., the network isn't accidentally segregating agents by type).


In [ ]:
def matrix_hash(mat: np.ndarray) -> str:
    return hashlib.sha256(np.ascontiguousarray(mat).tobytes()).hexdigest()


def matrix_has_cross_type_links(mat: np.ndarray) -> bool:
    if mat.size == 0:
        return False
    off_diag = mat.copy()
    np.fill_diagonal(off_diag, 0)
    return bool((off_diag > 0).any())


rows = []
for _, row in df_runs[df_runs["setting"] == "random_experts"].iterrows():
    A, _ = load_adjacency_matrix(row["run_path"])
    agent_data = load_agents_data(row["run_path"])
    agent_ids = sorted(agent_data["models"].keys())
    models_list = [agent_data["models"][aid] for aid in agent_ids]
    roles_list = [agent_data["roles"][aid] for aid in agent_ids]

    role_conn = build_connection_matrix(A, roles_list, ROLE_TO_IDS)
    model_conn = build_connection_matrix(A, models_list, MODEL_TO_IDS)

    rows.append({
        "graph_type": row["graph_type"], "statement_id": row["statement_id"], "seed": int(row["seed"]),
        "role_conn_sig": matrix_hash(role_conn), "model_conn_sig": matrix_hash(model_conn),
        "role_cross_links": matrix_has_cross_type_links(role_conn),
        "model_cross_links": matrix_has_cross_type_links(model_conn),
        "role_vs_model_matrix_diff": matrix_hash(role_conn) != matrix_hash(model_conn),
    })
re_conn = pd.DataFrame(rows)

uniq = (
    re_conn.groupby(["graph_type", "statement_id"])
    .agg(n_runs=("seed", "size"), unique_role_conn=("role_conn_sig", "nunique"),
         unique_model_conn=("model_conn_sig", "nunique"))
    .reset_index()
)
uniq_fail = uniq[
    (uniq["unique_role_conn"] != uniq["n_runs"]) | (uniq["unique_model_conn"] != uniq["n_runs"])
]
cross_fail = re_conn[(~re_conn["role_cross_links"]) | (~re_conn["model_cross_links"])]
same_matrix_fail = re_conn[~re_conn["role_vs_model_matrix_diff"]]

print("=== random_experts connectivity summary ===")
print(f"Uniqueness failures across seeds:        {len(uniq_fail)}")
print(f"Cross-type connectivity failures:        {len(cross_fail)}")
print(f"Role-vs-model identical-matrix failures: {len(same_matrix_fail)}")

if len(uniq_fail) == 0 and len(cross_fail) == 0 and len(same_matrix_fail) == 0:
    print("\nAll random_experts connectivity checks passed.")
